In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import PIL.Image as Image
from pathlib import Path
!pip install torchmetrics
from torchvision import models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import MulticlassF1Score
from torchmetrics import Accuracy
import torch.optim as optim
from torch import nn
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'{device} is available as our device')

In [ ]:
# Define the project root so later cells can resolve data and checkpoint paths reliably
from pathlib import Path
import zipfile

ROOT_DIR = None
EXTRACT_DIR = Path('/content/drive/MyDrive/plant-disease-multhead_extracted')
ALT_EXTRACT_DIR = Path('/content/plant-disease-multhead')
DRIVE_ROOT = Path('/content/drive/MyDrive')
ZIP_NAMES = ['plant-disease-multhead.zip', 'plant_disease_multhead.zip']

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Prefer an already-extracted copy if available.
    extracted_candidates = [
        EXTRACT_DIR,
        ALT_EXTRACT_DIR,
        DRIVE_ROOT / 'plant-disease-multhead',
        DRIVE_ROOT / 'plant_disease_multhead',
    ]

    for candidate in extracted_candidates:
        if candidate.exists() and (candidate / 'plant').exists():
            ROOT_DIR = str(candidate.resolve())
            break

    if ROOT_DIR is None:
        for zip_name in ZIP_NAMES:
            zip_path = DRIVE_ROOT / zip_name
            if zip_path.exists():
                print(f'Found project zip: {zip_path}')

                if EXTRACT_DIR.exists() and (EXTRACT_DIR / 'plant').exists():
                    ROOT_DIR = str(EXTRACT_DIR.resolve())
                    print(f'Using existing extracted folder: {ROOT_DIR}')
                    break

                print(f'Extracting zip to {EXTRACT_DIR}...')
                EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(EXTRACT_DIR)

                if (EXTRACT_DIR / 'plant-disease-multhead').exists():
                    ROOT_DIR = str((EXTRACT_DIR / 'plant-disease-multhead').resolve())
                elif (EXTRACT_DIR / 'plant').exists():
                    ROOT_DIR = str(EXTRACT_DIR.resolve())
                else:
                    ROOT_DIR = str(EXTRACT_DIR.resolve())
                break
except Exception as exc:
    print('Drive mount or zip extraction did not complete:', exc)
    ROOT_DIR = None

if ROOT_DIR is None:
    ROOT_DIR = str(Path.cwd().resolve())

print(f'Using project root: {ROOT_DIR}')

In [ ]:
class PlantDiseaseModel(nn.Module):
    def __init__(self, num_plant_classes: int, num_disease_classes: int):
        super(PlantDiseaseModel, self).__init__()
        base_model = models.mobilenet_v3_small(weights='DEFAULT')
        self.shared = nn.Sequential(
            base_model.features,
            base_model.avgpool
        )

        in_features = 576
        self.disease_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 1024),
            nn.Hardswish(),
            nn.Dropout(0.3, inplace=True),
            nn.Linear(1024, num_disease_classes)
        )

        self.plant_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_plant_classes)
        )

    def forward(self, x):
        features = self.shared(x)
        plant_outputs = self.plant_head(features)
        disease_outputs = self.disease_head(features)
        return plant_outputs, disease_outputs

Train mean = [0.46160856 0.55330724 0.3025134]
Train std = [0.16686249 0.17260724 0.15989958]

In [ ]:
transform = transforms.Compose([
    transforms.RandomAdjustSharpness(1.5),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4616, 0.5533, 0.3025), (0.1669, 0.1726, 0.1599))
])

In [ ]:
class CropDiseaseDataset(Dataset):
    def __init__(self, csv_file: str, transform=None, root_dir=None):
        super().__init__()
        df = pd.read_csv(csv_file)
        self.data = df.to_numpy()
        self.transform = transform
        self.root_dir = root_dir or os.getcwd()
        self.disease_to_idx = {name: i for i, name in enumerate(np.unique(self.data[:, 1]))}
        self.plant_to_idx = {name: i for i, name in enumerate(np.unique(self.data[:, 0]))}

    def __len__(self):
        return len(self.data)

    def _resolve_image_path(self, image_path):
        path = str(image_path).strip().replace('\\', '/')
        if not path:
            raise FileNotFoundError('Empty image path found in dataset.')

        if os.path.exists(path):
            return path

        # Only use the uploaded project structure in Colab.
        candidates = []
        if path.startswith('plant/'):
            candidates.append(os.path.join(self.root_dir, path))
            candidates.append(os.path.join(self.root_dir, path[len('plant/'):]))
        else:
            candidates.append(os.path.join(self.root_dir, path))
            candidates.append(os.path.join(self.root_dir, 'plant', path))
            candidates.append(os.path.join(self.root_dir, 'plant', os.path.basename(path)))

        for candidate in candidates:
            if os.path.exists(candidate):
                return candidate

        filename = os.path.basename(path)
        for root, _, files in os.walk(self.root_dir):
            if filename in files:
                return os.path.join(root, filename)

        raise FileNotFoundError(f'Could not find image: {image_path}')

    def __getitem__(self, index: int):
        image_path = self._resolve_image_path(self.data[index, 2])
        image = Image.open(image_path).convert('RGB')
        crop_type = self.plant_to_idx[self.data[index, 0]]
        disease_type = self.disease_to_idx[self.data[index, 1]]
        image = self.transform(image)
        return image, torch.tensor(crop_type), torch.tensor(disease_type)

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split


def resolve_project_root():
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/content/drive/MyDrive/plant-disease-multhead'),
        Path('/content/drive/MyDrive/plant_disease_multhead'),
        Path('/content/drive/MyDrive'),
    ])

    for candidate in candidates:
        if not candidate.exists():
            continue
        if (candidate / 'plant').exists():
            return candidate.resolve()
        if (candidate / 'plant_extracted' / 'plant').exists():
            return candidate.resolve()
        if (candidate / 'data').exists() and (candidate / 'src').exists():
            return candidate.resolve()

    return cwd


ROOT_DIR = str(resolve_project_root())
BASE_PATH = Path(ROOT_DIR)
print(f'Using project root: {BASE_PATH}')

image_root = None
for candidate in [BASE_PATH / 'plant', BASE_PATH / 'plant_extracted' / 'plant']:
    if candidate.exists():
        image_root = candidate
        break

if image_root is None:
    raise FileNotFoundError(f'Could not find image root under {BASE_PATH}')

records = []
for plant_dir in sorted(image_root.iterdir()):
    if not plant_dir.is_dir():
        continue

    for disease_dir in sorted(plant_dir.iterdir()):
        if not disease_dir.is_dir():
            continue

        for image_file in sorted(disease_dir.iterdir()):
            if image_file.is_file():
                rel_path = image_file.relative_to(BASE_PATH).as_posix()
                records.append({
                    'plant': plant_dir.name,
                    'disease': disease_dir.name.split('__')[-1],
                    'image': rel_path,
                })

print(f'Found {len(records)} images')

image_df = pd.DataFrame(records)

raw_csv_path = BASE_PATH / 'data' / 'crop_disease_labels.csv'
raw_csv_path.parent.mkdir(parents=True, exist_ok=True)
image_df.to_csv(raw_csv_path, index=False)
print(f'Saved rebuilt dataset to {raw_csv_path}')

train_df, temp_df = train_test_split(image_df, stratify=image_df['disease'], test_size=0.2, random_state=42)
test_df, val_df = train_test_split(temp_df, stratify=temp_df['disease'], test_size=0.5, random_state=42)

train_csv_path = BASE_PATH / 'data' / 'train.csv'
test_csv_path = BASE_PATH / 'data' / 'test.csv'
val_csv_path = BASE_PATH / 'data' / 'val.csv'

train_df.to_csv(train_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)

print(f'Saved train.csv to {train_csv_path}')
print(f'Saved test.csv to {test_csv_path}')
print(f'Saved val.csv to {val_csv_path}')

train_data = CropDiseaseDataset(str(train_csv_path), transform=transform, root_dir=str(BASE_PATH))
test_data = CropDiseaseDataset(str(test_csv_path), transform=transform, root_dir=str(BASE_PATH))
val_data = CropDiseaseDataset(str(val_csv_path), transform=transform, root_dir=str(BASE_PATH))

train_loader = DataLoader(train_data, batch_size=30, shuffle=True, pin_memory=False)
test_loader = DataLoader(test_data, batch_size=32)
val_loader = DataLoader(val_data, batch_size=32)

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')

In [ ]:
NUM_PLANT_CLASSES = 5
NUM_DISEASE_CLASSES = 26

model = PlantDiseaseModel(NUM_PLANT_CLASSES, NUM_DISEASE_CLASSES)
model = model.to(device)

In [ ]:
EPOCHS = 80

optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

f1score_plant = MulticlassF1Score(num_classes=NUM_PLANT_CLASSES, average='macro').to(device)
f1score_disease = MulticlassF1Score(num_classes=NUM_DISEASE_CLASSES, average='macro').to(device)

accuracy_plant = Accuracy(task='multiclass', num_classes=NUM_PLANT_CLASSES).to(device)
accuracy_disease = Accuracy(task='multiclass', num_classes=NUM_DISEASE_CLASSES).to(device)

In [ ]:
checkpoint_dir = os.path.join(ROOT_DIR, 'models')
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'checkpoint_*.pth')))
best_checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pth')

start_epoch = 1
if os.path.exists(best_checkpoint_path):
    latest_checkpoint = best_checkpoint_path
    print(f'Loading best checkpoint from {latest_checkpoint}')
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        state_dict = checkpoint['model']
        start_epoch = checkpoint.get('epoch', 0) + 1
    elif isinstance(checkpoint, dict) and any(k.startswith('module.') for k in checkpoint.keys()):
        state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=False)
elif checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    print(f'Loaded checkpoint from {latest_checkpoint}')
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        state_dict = checkpoint['model']
        start_epoch = checkpoint.get('epoch', 0) + 1
    elif isinstance(checkpoint, dict) and any(k.startswith('module.') for k in checkpoint.keys()):
        state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=False)
    try:
        start_epoch = int(os.path.basename(latest_checkpoint).split('_')[1].split('.')[0]) + 1
    except (IndexError, ValueError):
        start_epoch = 1
else:
    print('No previous checkpoint found. Starting training from scratch.')

In [ ]:
best_val_score = -float('inf')
model.train()
for epoch in range(start_epoch, EPOCHS + 1):
    running_loss = 0.0
    for image, crop, disease in train_loader:
        optimizer.zero_grad()
        image = image.to(device)
        crop = crop.to(device)
        disease = disease.to(device)

        plant_outputs, disease_outputs = model(image)
        plant_loss = criterion(plant_outputs, crop.long())
        disease_loss = criterion(disease_outputs, disease.long())

        f1score_plant.update(plant_outputs, crop.long())
        accuracy_plant.update(plant_outputs, crop.long())
        f1score_disease.update(disease_outputs, disease.long())
        accuracy_disease.update(disease_outputs, disease.long())

        combined_loss = plant_loss + disease_loss
        combined_loss.backward()
        running_loss += combined_loss.item()
        optimizer.step()

    f1_plant = f1score_plant.compute().item()
    acc_plant = accuracy_plant.compute().item()
    f1_disease = f1score_disease.compute().item()
    acc_disease = accuracy_disease.compute().item()
    print(f'Epoch|{epoch} Loss| {running_loss / len(train_loader):.2f} | Plant F1 Score| {f1_plant * 100:.2f}% | Disease F1 Score| {f1_disease * 100:.2f}% | Plant Accuracy| {acc_plant * 100:.2f}% | Disease Accuracy| {acc_disease * 100:.2f}%')

    f1score_plant.reset()
    f1score_disease.reset()
    accuracy_plant.reset()
    accuracy_disease.reset()

    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for image, crop, disease in val_loader:
            image = image.to(device)
            crop = crop.to(device)
            disease = disease.to(device)

            plant_outputs, disease_outputs = model(image)
            plant_loss = criterion(plant_outputs, crop.long())
            disease_loss = criterion(disease_outputs, disease.long())
            val_loss += (plant_loss + disease_loss).item()

            f1score_plant.update(plant_outputs, crop.long())
            accuracy_plant.update(plant_outputs, crop.long())
            f1score_disease.update(disease_outputs, disease.long())
            accuracy_disease.update(disease_outputs, disease.long())

        acc_plant = accuracy_plant.compute().item()
        acc_disease = accuracy_disease.compute().item()
        f1_plant = f1score_plant.compute().item()
        f1_disease = f1score_disease.compute().item()
        val_score = (f1_plant + f1_disease + acc_plant + acc_disease) / 4
        print('------------- VALIDATION -------------')
        print(f'Plant F1 Score-- {f1_plant * 100:.2f}% -- Disease Accuracy-- {f1_disease * 100:.2f}% -- Plant Accuracy-- {acc_plant * 100:.2f}% -- Disease Accuracy-- {acc_disease * 100:.2f}%')
        print(f'Validation score: {val_score:.4f}')

        if val_score > best_val_score:
            best_val_score = val_score
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'val_score': val_score
            }, best_checkpoint_path)
            print(f'New best model saved to {best_checkpoint_path}')

        f1score_plant.reset()
        f1score_disease.reset()
        accuracy_plant.reset()
        accuracy_disease.reset()

    if epoch % 5 == 0:
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, f'checkpoint_{epoch}.pth'))
        print('Periodic checkpoint saved to models folder')

model.train()
torch.save(model.state_dict(), os.path.join(ROOT_DIR, 'multi_head_plant_disease.pth'))